### Imports

In [ ]:
from pathlib import Path

import pandas as pd

### Read the data from the Excel file

In [ ]:
RAW_PATH = Path("../data/raw")
assert RAW_PATH.exists(), "Create raw folder and add the necessary files inside."

In [ ]:
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(exist_ok=True)

In [ ]:
SIMULADOR_FILE = RAW_PATH / "Simulador Bloco e Subbloco_v2.xlsx"
assert SIMULADOR_FILE.exists(), "Missing Simulador file add to data/raw folder."

In [ ]:
excel = pd.ExcelFile(SIMULADOR_FILE, engine="calamine")
print(excel.sheet_names)

# Data cleaning steps

1. Extract 'Calendário FV' sheet from the provided Excel file.
    - extract sheet by sheet name
    - drop empty columns
    - drop index column
2. Extract 'Base Demanda' sheet from the provided Excel file.
    - extract sheet by sheet name
    - filter columns (not calculated columns)
    - merge with 'Calendário FV' sheet to get the 'Dia Captacao' column

### Extract calendar data from the 'Calendário FV' sheet

In [ ]:
df_calendario = excel.parse("Calendário FV")

In [ ]:
# drop empty columns
df_calendario = df_calendario.dropna(how="all", axis=1)

In [ ]:
df_calendario = df_calendario.drop(columns=["#"])

In [ ]:
# export csv
output_path = PROCESSED_PATH / "calendario_fv.csv"
df_calendario.to_csv(output_path, index=False)

### Extract demand from  'Base Demanda' sheet

Notes:
- Dia Captacao depends on data from calendar that seems to not be present in the 'Calendário FV' sheet. It is possible that this data is in another sheet or file. Further investigation is needed to locate the source of this information. Or even if this is needed. Initial thought it that it's not needed.

In [ ]:
df_demand = excel.parse("Base Demanda")

In [ ]:
DEMAND_COLUMNS_NEEDED = [
    "data_pedido",
    "cd_setor",
    "cd_cd",
    "nm_ciclo",
    "aa_ciclo",
    "total_pedidos",
    "total_volumes",
    "total_itens",
]
df_demand = df_demand[DEMAND_COLUMNS_NEEDED]

In [ ]:
df_demand["ciclo"] = df_demand["aa_ciclo"].astype(str) + df_demand["nm_ciclo"].astype(
    str
).str.zfill(2)

In [ ]:
df_calendario["CICLOS"] = df_calendario["CICLOS"].astype(str)

In [ ]:
output_file = PROCESSED_PATH / "demanda_with_calendar.csv"
df = df_demand.merge(df_calendario, left_on=["ciclo", "cd_setor"], right_on=["CICLOS", "COD SETOR"])
df.to_csv(output_file, index=False)

# Exploration

In [ ]:
# I expect that every cycle start and end at the same day if the sector is in
# the same block + sublock
# this is the code to check this hypothesis is correct
is_consistent = (
    df_calendario.groupby(["BLOCO", "SUB BLOCO", "CICLOS"])[["Dt Abertura", "Dt Fechamento"]]
    .nunique()
    .eq(1)
    .all()
    .all()
)

print(f"Expectation holds: {is_consistent}")

In [ ]:
# which blocks/subblocks/cycles diverge (and which sectors differ):
# Count unique start and end dates per block + subblock + cycle
cycle_dates_summary = df_calendario.groupby(["BLOCO", "SUB BLOCO", "CICLOS"])[
    ["Dt Abertura", "Dt Fechamento"]
].nunique()

# Filter for groups that have more than 1 distinct date
discrepancies = cycle_dates_summary[
    (cycle_dates_summary["Dt Abertura"] > 1) | (cycle_dates_summary["Dt Fechamento"] > 1)
]

if discrepancies.empty:
    print(
        "Confirmed: Every cycle starts and ends on the same day for "
        "all sectors in the same block + sublock."
    )
else:
    print(
        f"Found {len(discrepancies)} (BLOCO, SUB BLOCO, CICLOS) combination(s) "
        "with diverging dates:\n"
    )
    display(discrepancies)

    # Inspect the exact sectors and dates causing the mismatch
    conflicting_rows = df_calendario.merge(
        discrepancies.reset_index()[["BLOCO", "SUB BLOCO", "CICLOS"]],
        on=["BLOCO", "SUB BLOCO", "CICLOS"],
    )
    display(
        conflicting_rows[
            [
                "BLOCO",
                "SUB BLOCO",
                "CICLOS",
                "COD SETOR",
                "Dt Abertura",
                "Dt Fechamento",
                "Qtde dias",
            ]
        ].sort_values(
            by=[
                "CICLOS",
                "BLOCO",
                "SUB BLOCO",
                "Qtde dias",
                "COD SETOR",
            ]
        )
    )